# Aktivitás részletesen
Állítsd be az `ACTIVITY_ID`-t a lenti listából (vagy hagyd `None`-on a legutóbbihoz), majd **Run All**.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from mygarmin.analysis import list_activities, pick_activity, lap_stats

DATA = Path.cwd().parent / "data" if Path.cwd().name == "notebooks" else Path.cwd() / "data"
DB = DATA / "garmin.db"
pd.set_option("display.max_columns", 50)

list_activities(DB).drop(columns="raw_dir")

In [ ]:
ACTIVITY_ID = None  # pl. 24398713557; None = legutóbbi

act = pick_activity(DB, ACTIVITY_ID)
s, d, w = act.summary, act.details_summary, act.weather


def fmt_min(sec):
    return f"{int(sec // 60)}:{int(sec % 60):02d}" if sec else None


overview = {
    "Név": act.name,
    "Típus": act.type,
    "Kezdés": s.get("startTimeLocal"),
    "Időtartam (p:mp)": fmt_min(s.get("duration")),
    "Mozgásban (p:mp)": fmt_min(s.get("movingDuration")),
    "Távolság (km)": round((s.get("distance") or 0) / 1000, 2),
    "Szintemelkedés / süllyedés (m)": f"{s.get('elevationGain')} / {s.get('elevationLoss')}" if s.get("elevationGain") else None,
    "Átlag / max pulzus": f"{s.get('averageHR')} / {s.get('maxHR')}",
    "Kalória (kcal)": s.get("calories"),
    "Aerob / anaerob edzéshatás": f"{d.get('trainingEffect')} / {d.get('anaerobicTrainingEffect')}",
    "Edzéshatás címke": d.get("trainingEffectLabel"),
    "Edzésterhelés": d.get("activityTrainingLoad"),
    "Body Battery változás": d.get("differenceBodyBattery"),
    "Intenzív percek (mérsékelt / erős)": f"{d.get('moderateIntensityMinutes')} / {d.get('vigorousIntensityMinutes')}",
    "Körök száma": len(act.laps),
}
pd.Series({k: v for k, v in overview.items() if v not in (None, "None / None", "")}, name="érték").to_frame()

## Idősorok
Függőleges vonalak: körhatárok. Szürke sáv: pihenő (REST / RECOVERY) kör.

In [ ]:
LABELS = {
    "hr": "Pulzus (bpm)", "speed_kmh": "Sebesség (km/h)", "elevation_m": "Magasság (m)",
    "cadence_spm": "Kadencia (spm)", "kcal_per_min": "Égetés (kcal/perc)", "body_battery": "Body Battery",
    "grit": "Grit", "flow": "Flow",
}
series = act.series
cols = [c for c in LABELS if c in series and series[c].notna().any() and series[c].max() > 0]
laps = lap_stats(act)

if not cols:
    print("Nincs idősor ehhez az aktivitáshoz.")
else:
    fig, axes = plt.subplots(len(cols), 1, figsize=(13, 2.3 * len(cols)), sharex=True, squeeze=False)
    for ax, c in zip(axes[:, 0], cols):
        ax.plot(series["elapsed_min"], series[c], lw=1)
        ax.set_ylabel(LABELS[c])
        ax.grid(alpha=0.3)
        if len(laps) > 1 and "start_min" in laps:
            for _, lap in laps.iterrows():
                ax.axvline(lap["start_min"], color="k", alpha=0.15, lw=0.8)
                if lap["intensity"] in ("REST", "RECOVERY"):
                    ax.axvspan(lap["start_min"], lap["start_min"] + lap["duration_s"] / 60, color="grey", alpha=0.15)
    axes[-1, 0].set_xlabel("Eltelt idő (perc)")
    fig.suptitle(f"{act.name} ({act.type})")
    plt.tight_layout()

## Pulzuszónák

In [ ]:
z = act.hr_zones
if z.empty:
    print("Nincs zóna adat.")
else:
    colors = ["#9e9e9e", "#2196f3", "#4caf50", "#ff9800", "#f44336"]
    ax = z.plot.barh(x="zone", y="minutes", color=colors[: len(z)], legend=False, figsize=(8, 3))
    ax.set_yticklabels([f"Z{r.zone} (≥{r.from_bpm})" for r in z.itertuples()])
    ax.set_xlabel("perc")
    for i, m in enumerate(z["minutes"]):
        ax.text(m, i, f" {m:.1f} p ({m / z['minutes'].sum():.0%})", va="center")
    ax.invert_yaxis()

## Körök
`hr_drop` = a körön belüli pulzuscsúcs mínusz a kör végi pulzus. Pihenő köröknél ez a **pulzus-visszaállás**.
(A mért csúcs gyakran már a pihenő körbe esik: a pulzus fokozatosan reagál a terhelésre, a csukló-optikai szenzor simít/késik, és mászásnál a megfeszült alkar is torzíthatja a mérést. Mellkaspánttal pontosabb.)

In [ ]:
if laps.empty:
    print("Nincsenek körök.")
else:
    view = laps.assign(
        perc=(laps["duration_s"] / 60).round(2),
        km=(laps["distance_m"] / 1000).round(2),
    )[["lap", "intensity", "perc", "km", "avg_hr", "max_hr", "hr_start", "hr_peak", "hr_end", "hr_drop", "calories"]
      if "hr_drop" in laps else ["lap", "intensity", "perc", "km", "avg_hr", "max_hr", "calories"]]
    display(view)

    if len(laps) > 1:
        palette = {"ACTIVE": "#f44336", "INTERVAL": "#f44336", "REST": "#90caf9", "RECOVERY": "#90caf9",
                   "WARMUP": "#ffcc80", "COOLDOWN": "#a5d6a7"}
        fig, ax = plt.subplots(figsize=(13, 3.5))
        ax.bar(laps["lap"], laps["duration_s"] / 60, color=[palette.get(i, "#bdbdbd") for i in laps["intensity"]])
        ax.set_ylabel("Kör hossza (perc)")
        ax.set_xlabel("Kör")
        ax2 = ax.twinx()
        ax2.plot(laps["lap"], laps["avg_hr"], "k.-", label="átlag pulzus")
        ax2.set_ylabel("Átlag pulzus")
        ax.set_title("Körök – piros: aktív, kék: pihenő, narancs: bemelegítés, zöld: levezetés")

In [ ]:
rest = laps[laps["intensity"].isin(["REST", "RECOVERY"])] if "hr_drop" in laps else pd.DataFrame()
if len(rest) >= 2:
    ax = rest.plot(x="lap", y="hr_drop", kind="bar", legend=False, figsize=(10, 3), color="#1976d2")
    ax.set_ylabel("Pulzuscsökkenés csúcsról (bpm)")
    ax.set_title(f"Pulzus-visszaállás pihenőkben – átlag {rest['hr_drop'].mean():.0f} bpm / kör")
else:
    print("Nincs elég pihenő kör a visszaállás-elemzéshez.")

## Útvonal
Csak GPS-es aktivitásnál. Szín: pulzus.

In [ ]:
if not act.has_gps:
    print("Ennek az aktivitásnak nincs GPS útvonala.")
else:
    g = series.dropna(subset=["lat", "lon"])
    fig, ax = plt.subplots(figsize=(8, 8))
    sc = ax.scatter(g["lon"], g["lat"], c=g.get("hr"), cmap="plasma", s=4)
    ax.plot(g["lon"].iloc[0], g["lat"].iloc[0], "go", ms=10, label="start")
    ax.plot(g["lon"].iloc[-1], g["lat"].iloc[-1], "rs", ms=10, label="cél")
    ax.set_aspect(1 / __import__("math").cos(__import__("math").radians(g["lat"].mean())))
    ax.legend()
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(sc, ax=ax, label="Pulzus (bpm)", shrink=0.7)

    if "elevation_m" in series and "distance_m" in series:
        fig, ax = plt.subplots(figsize=(13, 3))
        ax.fill_between(series["distance_m"] / 1000, series["elevation_m"], series["elevation_m"].min(), alpha=0.4)
        ax.set_xlabel("km"); ax.set_ylabel("Magasság (m)"); ax.set_title("Magassági profil")

## Időjárás

In [ ]:
if not w or w.get("temp") is None:
    print("Nincs időjárás adat.")
else:
    f2c = lambda f: round((f - 32) * 5 / 9, 1) if f is not None else None  # Garmin °F-ben adja
    pd.Series({
        "Hőmérséklet (°C)": f2c(w.get("temp")),
        "Hőérzet (°C)": f2c(w.get("apparentTemp")),
        "Páratartalom (%)": w.get("relativeHumidity"),
        "Szél (km/h)": round(w["windSpeed"] * 1.609, 1) if w.get("windSpeed") is not None else None,
        "Szélirány": w.get("windDirectionCompassPoint"),
        "Idő": (w.get("weatherTypeDTO") or {}).get("desc"),
        "Állomás": (w.get("weatherStationDTO") or {}).get("name"),
    }, name="érték").to_frame()

## Eredeti FIT fájl
Amit a Garmin feldolgozott adatai nem tartalmaznak, az itt lehet. A `session` üzenet az aktivitás teljes összesítője.

In [ ]:
fit = act.fit_messages()
display(pd.Series({k: len(v) for k, v in fit.items()}, name="üzenetek száma").sort_values(ascending=False).to_frame())
session = pd.Series(fit.get("session", [{}])[0], name="session").dropna()
session.to_frame()